# model2 public inference
Attach the competition inputs and the saved training notebook output. Run all cells. Elapsed time includes loading artifacts, feature processing, scoring and CSV writing.

In [ ]:
from pathlib import Path
import time, joblib, sys
WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else next(p/'Models/Emad/model3e_plus_cache' for p in [Path.cwd(),*Path.cwd().parents] if (p/'Models/Emad').is_dir())
WORK.mkdir(parents=True,exist_ok=True)
started=time.perf_counter()
def find_bundle(name):
    candidates=[WORK/name]+list(Path('/kaggle/input').rglob(name))
    hits=[p for p in candidates if p.is_dir() and (p/(name.split('_')[0]+'.joblib')).is_file()]
    if not hits:
        from zipfile import ZipFile
        import shutil
        archives=list(Path('/kaggle/input').rglob(name+'.zip'))
        if len(archives)==1:
            dst=WORK/'loaded_artifacts'/name
            dst.mkdir(exist_ok=True,parents=True)
            model=name.split('_')[0]
            filenames=[model+'.joblib',model+'_runtime.py']
            if model=='model8': filenames.append('public_sift_pairs.npz')
            with ZipFile(archives[0]) as z:
                for filename in filenames:
                    with z.open(name+'/'+filename) as src,(dst/filename).open('wb') as out:
                        shutil.copyfileobj(src,out)
            return dst
        # Kaggle can flatten an uploaded archive into the dataset directory.
        hits=[p.parent for p in Path('/kaggle/input').rglob(name.split('_')[0]+'.joblib') if (p.parent/(name.split('_')[0]+'_runtime.py')).is_file()]
    if len(hits)!=1: raise ValueError(f'Attach one artifact bundle for {name}. Found {hits}')
    return hits[0]
bundle=find_bundle('model2_artifacts')
exec((bundle/'model2_runtime.py').read_text())
params=joblib.load(bundle/'model2.joblib')
train_df,test_df,norm_features,sift_data,sample_path=inputs()
result=predict_public(params)
out=WORK/'submission_model2_hybrid_openset.csv'
result.to_csv(out,index=False)
print(f'{len(result)} rows in {time.perf_counter()-started:.2f} seconds. Saved {out}')